In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

In [3]:
# Helper function to generate random choices throughout the process/algorithm

# Create a circuit with one qubit and one classical bit to store after measurement
circuit = QuantumCircuit(1, 1)

# Add a Hadamard (H) gate on qubit 0, to create a superposition
circuit.h(0)

# Map the quantum measurement to the classical bits
circuit.measure([0], [0])

# Use BasicSimulator
backend = BasicSimulator()

def random_bit_generator():

    # Run simulation to get measurement
    qc_compiled = transpile(circuit, backend)
    job_sim = backend.run(qc_compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(qc_compiled)

    # Convert to integer before returning
    rand_bit = int(list(counts.keys())[0])
    return rand_bit

In [4]:
# Helper function to translate received qubit to bit with the selected basis

# Use BasicSimulator
backend_t = BasicSimulator()

def translate_qubit_to_bit(qubit, basis):

    # Create circuit for individual qubit
    circuit = QuantumCircuit(1, 1)

    # Prepare qubit to simulate the received qubit
    if qubit == 0:
        pass

    elif qubit == 1:
        circuit.x(0)

    elif qubit == "+":
        circuit.h(0)

    elif qubit == "-":
        circuit.x(0)
        circuit.h(0)

    # Measurement

    # If basis is diagonal, apply H gate before measuring
    if basis == "d":
        circuit.h(0)

    # Measure
    circuit.measure([0], [0])

    # Run simulation to get measurement
    qc_compiled = transpile(circuit, backend_t)
    job_sim = backend_t.run(qc_compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(qc_compiled)

    # Convert to integer before returning
    measured_qubit = int(list(counts.keys())[0])
    return measured_qubit

In [5]:
# Step 1: Alice(sender) generate random sequence of bits

# List to store random generated bits
random_bit_sequence = []

for i in range(100): # Edit number in range to determine length of sequence
    random_bit_sequence.append(random_bit_generator())

# Check if its random
print("Random bit sequence:", random_bit_sequence)

Random bit sequence: [0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]


In [6]:
# Step 2: Alice represents each bit by qubit using one of two schemes
#         (a) represent 0 by |0> and 1 by |1> (the standard basis)
#         (b) represent 0 by |+> and 1 by |-> (the diagonal basis)
#         chosen randomly and independently for each bit
#
# Lets define 0 -> standard basis (s)
#             1 -> diagonal basis (d)

# List to store sequence of basis randomly chosen by sender
sender_basis_sequence = []

# Generate random sequence of basis
for i in range(len(random_bit_sequence)):
    basis = random_bit_generator()
    if basis == 0:
        sender_basis_sequence.append("s")
    else:
        sender_basis_sequence.append("d")

# List to store sequence of qubits translated from the random_bit_sequence
qubit_sequence = []

for i in range(len(sender_basis_sequence)):
    if sender_basis_sequence[i] == "s":
        if random_bit_sequence[i] == 0:
            qubit_sequence.append(0)
        else:
            qubit_sequence.append(1)
    else:
        if random_bit_sequence[i] == 0:
            qubit_sequence.append('+')
        else:
            qubit_sequence.append('-')

# Check if sequence of bit has been translated randomly and correctly
print("Basis sequence randomly selected by sender:", sender_basis_sequence)
print("Translated qubit sequence:", qubit_sequence)

Basis sequence randomly selected by sender: ['s', 'd', 'd', 's', 's', 'd', 's', 'd', 'd', 's', 's', 'd', 's', 'd', 'd', 's', 's', 'd', 'd', 's', 'd', 's', 'd', 's', 's', 's', 'd', 'd', 's', 'd', 's', 'd', 's', 's', 's', 'd', 's', 'd', 'd', 's', 'd', 'd', 's', 'd', 's', 's', 'd', 'd', 'd', 's', 'd', 's', 's', 'd', 's', 's', 's', 's', 's', 's', 'd', 's', 's', 's', 'd', 's', 'd', 'd', 's', 'd', 'd', 'd', 'd', 'd', 'd', 's', 's', 'd', 's', 's', 'd', 'd', 's', 's', 's', 'd', 'd', 'd', 'd', 's', 'd', 's', 's', 'd', 's', 'd', 'd', 'd', 's', 'd']
Translated qubit sequence: [0, '+', '-', 0, 1, '-', 1, '-', '-', 1, 0, '-', 1, '-', '-', 1, 0, '+', '-', 0, '-', 0, '-', 0, 1, 1, '+', '+', 1, '+', 1, '-', 0, 1, 1, '+', 0, '-', '+', 0, '+', '-', 0, '-', 1, 1, '-', '-', '+', 1, '-', 0, 1, '-', 0, 1, 0, 1, 0, 0, '-', 0, 1, 1, '-', 0, '-', '-', 1, '-', '-', '+', '-', '+', '-', 0, 0, '-', 1, 1, '+', '+', 1, 1, 0, '-', '-', '+', '+', 0, '+', 0, 0, '+', 0, '+', '+', '-', 0, '+']


In [7]:
# Step 3: Bob(receiver) received sequence of qubits
#
#         For each qubit he makes a
#         measurement in either the standard or diagonal basis,
#         chosen randomly and independently for each qubit.
#
# Remember, 0 -> standard basis (s)
#           1 -> diagonal basis (d)

# List to store sequence of basis randomly chosen by receiver
receiver_basis_sequence = []

# Generate random sequence of basis
for i in range(len(random_bit_sequence)):
    basis = random_bit_generator()
    if basis == 0:
        receiver_basis_sequence.append("s")
    else:
        receiver_basis_sequence.append("d")


# List to store sequence of bits translated from qubit_sequence with receiver_basis_sequence
translated_bit_sequence = []

# Translate qubit_sequence with receiver_basis_sequence
for i in range(len(qubit_sequence)):
    translated_bit = translate_qubit_to_bit(qubit_sequence[i], receiver_basis_sequence[i])
    translated_bit_sequence.append(translated_bit)


# Check if qubit_sequence has been translated
print("Basis sequence randomly selected by receiver:", receiver_basis_sequence)
print("Translated bit sequence:", translated_bit_sequence)

Basis sequence randomly selected by receiver: ['d', 's', 's', 's', 's', 's', 'd', 'd', 'd', 's', 's', 's', 's', 's', 'd', 's', 's', 'd', 's', 'd', 's', 'd', 'd', 'd', 's', 's', 's', 's', 's', 'd', 's', 'd', 's', 's', 's', 's', 'd', 's', 's', 's', 'd', 's', 's', 's', 'd', 's', 'd', 'd', 's', 's', 'd', 'd', 'd', 'd', 's', 'd', 's', 'd', 's', 's', 'd', 's', 'd', 'd', 'd', 's', 'd', 's', 'd', 'd', 's', 's', 's', 's', 's', 'd', 'd', 's', 's', 's', 'd', 'd', 'd', 'd', 's', 'd', 'd', 'd', 'd', 's', 'd', 's', 'd', 'd', 'd', 's', 'd', 's', 's', 'd']
Translated bit sequence: [1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]


In [8]:
# Step 4: Alice and Bob publicly compare their sequence of basis choice

comparison = [sender == receiver for sender, receiver in zip(sender_basis_sequence, receiver_basis_sequence)]

# Check if sequence of basis has been compared correctly
print("List of comparison:", comparison)

List of comparison: [False, False, False, True, True, False, False, True, True, True, True, False, True, False, True, True, True, True, False, False, False, False, True, False, True, True, False, False, True, True, True, True, True, True, True, False, False, False, False, True, True, False, True, False, False, True, True, True, False, True, True, False, False, True, True, False, True, False, True, True, True, True, False, False, True, True, True, False, False, True, False, False, False, False, False, False, False, False, True, True, True, True, False, False, True, True, True, True, True, True, True, True, False, True, False, False, True, False, True, True]


In [9]:
# Step 5: Both Alice and Bob extract the shared binary sequence

# Extract shared binary sequence from their own individual bit_sequence
# based on the positions where both of their basis_sequence matches

# Alice extract the shared binary sequence from her own initial random_bit_sequence
shared_bit_sequence_sender = [bit for bit, basis_matched in zip(random_bit_sequence, comparison) if basis_matched]

# Bob extract the shared binary sequence from his own translated_bit_sequence
shared_bit_sequence_receiver = [bit for bit, basis_matched in zip(translated_bit_sequence, comparison) if basis_matched]

# Check if shared binary sequence on both parties are matching
print("Extracted shared binary sequence by sender   :", shared_bit_sequence_sender)
print("Extracted shared binary sequence by receiver :", shared_bit_sequence_receiver)
print("Matches:", shared_bit_sequence_sender == shared_bit_sequence_receiver)

Extracted shared binary sequence by sender   : [0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Extracted shared binary sequence by receiver : [0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Matches: True


In [10]:
# Step 6: Alice and Bob verify their shared key by publicly comparing a subset of bits

# They sacrifice some bits from the shared key to check for any mismatches.
# In this plain (no attacker) case, there should be 0% error rate.

# Determine how many bits to sacrifice for verification
num_check_bits = len(shared_bit_sequence_sender) // 4 # 25% of the shared key

# Alice and Bob publicly compare the first num_check_bits of their shared sequences
sender_check = shared_bit_sequence_sender[:num_check_bits]
receiver_check = shared_bit_sequence_receiver[:num_check_bits]

# Count mismatches
mismatches = sum(a != b for a, b in zip(sender_check, receiver_check))
error_rate = mismatches / num_check_bits

print("Bits sacrificed for verification :", num_check_bits)
print("Sender's check bits              :", sender_check)
print("Receiver's check bits            :", receiver_check)
print("Mismatches                       :", mismatches)
print("Error rate                       :", error_rate)

# Define a threshold, if error rate exceeds it, an attack is suspected
error_threshold = 0.2  # 20%

if error_rate > error_threshold:
    print("\nWARNING: Error rate exceeds threshold. An attacker may be present.")
else:
    print("\nPASS: No attack detected. Proceeding with the remaining shared key.")

# Remove the sacrificed check bits for the final usable key
final_key_sender = shared_bit_sequence_sender[num_check_bits:]
final_key_receiver = shared_bit_sequence_receiver[num_check_bits:]

print("\nFinal key (sender)   :", final_key_sender)
print("Final key (receiver) :", final_key_receiver)
print("Keys match           :", (final_key_sender == final_key_receiver))

Bits sacrificed for verification : 14
Sender's check bits              : [0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1]
Receiver's check bits            : [0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1]
Mismatches                       : 0
Error rate                       : 0.0

PASS: No attack detected. Proceeding with the remaining shared key.

Final key (sender)   : [1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Final key (receiver) : [1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Keys match           : True


In [11]:
# Demonstration of the BB84 protocol, without an attacker done.